In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class SecureConfigurationChecker:
    """
    SecureConfigurationChecker

    This checker evaluates whether a research software artifact avoids
    insecure default configurations and unsafe operational patterns.

    It checks for:
    - hard-coded secrets or credentials
    - API keys or tokens
    - debug mode enabled
    - plaintext HTTP URLs
    - disabled SSL/TLS verification
    - unsafe subprocess shell usage
    - unsafe deserialization patterns
    - privileged Docker/container settings
    - broad file permissions
    - committed environment files

    Formal idea:
        secureConfiguration : A → {True, False}
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        minimum_score=5,
        max_high_severity_findings=0,
        max_medium_severity_findings=5
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.minimum_score = minimum_score
        self.max_high_severity_findings = max_high_severity_findings
        self.max_medium_severity_findings = max_medium_severity_findings

        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def read_text_file(self, path, max_chars=200000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def should_scan_file(self, path):
        excluded_dirs = {
            ".git",
            "__pycache__",
            ".pytest_cache",
            ".mypy_cache",
            "node_modules",
            ".venv",
            "venv",
            "build",
            "dist"
        }

        excluded_extensions = {
            ".png", ".jpg", ".jpeg", ".gif", ".pdf", ".zip", ".tar", ".gz",
            ".whl", ".so", ".dll", ".exe", ".bin", ".pkl", ".pickle",
            ".npy", ".npz", ".parquet", ".feather", ".h5", ".hdf5"
        }

        for part in path.parts:
            if part in excluded_dirs:
                return False

        if path.suffix.lower() in excluded_extensions:
            return False

        try:
            if path.stat().st_size > 2 * 1024 * 1024:
                return False
        except OSError:
            return False

        return True

    def collect_scan_files(self, repo_dir):
        scan_extensions = {
            ".py", ".ipynb", ".js", ".ts", ".json", ".yaml", ".yml",
            ".toml", ".ini", ".cfg", ".env", ".sh", ".bash", ".dockerfile",
            ".md", ".rst", ".txt"
        }

        scan_names = {
            "dockerfile",
            "docker-compose.yml",
            "docker-compose.yaml",
            ".env",
            ".env.example",
            ".env.sample",
            "settings.py",
            "config.py",
            "config.yml",
            "config.yaml",
            "secrets.yml",
            "secrets.yaml"
        }

        files = []

        for root, dirs, filenames in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv",
                    "build",
                    "dist"
                }
            ]

            for filename in filenames:
                path = Path(root) / filename
                lower_name = filename.lower()

                if lower_name in scan_names or path.suffix.lower() in scan_extensions:
                    if self.should_scan_file(path):
                        files.append(path)

        return files

    def detect_findings_in_file(self, repo_dir, path):
        text = self.read_text_file(path)
        findings = []

        try:
            relative = str(path.relative_to(repo_dir))
        except Exception:
            relative = str(path)

        patterns = [
            {
                "name": "possible_hardcoded_password",
                "severity": "high",
                "regex": r"(?i)(password|passwd|pwd)\s*[:=]\s*['\"][^'\"]{6,}['\"]",
                "description": "Possible hard-coded password"
            },
            {
                "name": "possible_api_key",
                "severity": "high",
                "regex": r"(?i)(api[_-]?key|apikey|access[_-]?key|secret[_-]?key)\s*[:=]\s*['\"][A-Za-z0-9_\-]{16,}['\"]",
                "description": "Possible hard-coded API key or access key"
            },
            {
                "name": "possible_token",
                "severity": "high",
                "regex": r"(?i)(token|auth[_-]?token|bearer)\s*[:=]\s*['\"][A-Za-z0-9_\-\.]{20,}['\"]",
                "description": "Possible hard-coded token"
            },
            {
                "name": "private_key_material",
                "severity": "high",
                "regex": r"-----BEGIN (RSA |DSA |EC |OPENSSH )?PRIVATE KEY-----",
                "description": "Private key material detected"
            },
            {
                "name": "aws_access_key_like_value",
                "severity": "high",
                "regex": r"AKIA[0-9A-Z]{16}",
                "description": "AWS access-key-like value detected"
            },
            {
                "name": "debug_enabled",
                "severity": "medium",
                "regex": r"(?i)(debug\s*=\s*true|debug:\s*true|DEBUG\s*=\s*True|app\.run\(.*debug\s*=\s*True)",
                "description": "Debug mode appears enabled"
            },
            {
                "name": "plaintext_http_url",
                "severity": "medium",
                "regex": r"http://[A-Za-z0-9\.\-_:\/\?\=&%#]+",
                "description": "Plaintext HTTP URL detected"
            },
            {
                "name": "ssl_verification_disabled",
                "severity": "high",
                "regex": r"(?i)(verify\s*=\s*False|ssl_verify\s*=\s*false|rejectUnauthorized\s*:\s*false|NODE_TLS_REJECT_UNAUTHORIZED\s*=\s*0)",
                "description": "SSL/TLS verification appears disabled"
            },
            {
                "name": "unsafe_subprocess_shell_true",
                "severity": "medium",
                "regex": r"subprocess\.(run|call|Popen|check_call|check_output)\(.*shell\s*=\s*True",
                "description": "subprocess used with shell=True"
            },
            {
                "name": "unsafe_eval_exec",
                "severity": "medium",
                "regex": r"(?<![A-Za-z0-9_])(eval|exec)\s*\(",
                "description": "Potential unsafe eval/exec usage"
            },
            {
                "name": "unsafe_pickle_load",
                "severity": "medium",
                "regex": r"(pickle\.load|pickle\.loads|joblib\.load)",
                "description": "Potential unsafe deserialization usage"
            },
            {
                "name": "yaml_unsafe_load",
                "severity": "medium",
                "regex": r"yaml\.load\s*\(",
                "description": "Potential unsafe yaml.load usage"
            },
            {
                "name": "broad_permission_chmod",
                "severity": "medium",
                "regex": r"(chmod\s+777|os\.chmod\(.*0o777)",
                "description": "Broad file permission detected"
            },
            {
                "name": "privileged_container",
                "severity": "high",
                "regex": r"(?i)(--privileged|privileged:\s*true|network_mode:\s*host|--net=host)",
                "description": "Privileged container or host network setting detected"
            },
            {
                "name": "sudo_usage_in_script",
                "severity": "low",
                "regex": r"(?m)^\s*sudo\s+",
                "description": "sudo usage detected in script/configuration"
            }
        ]

        for pattern in patterns:
            matches = re.finditer(pattern["regex"], text, flags=re.MULTILINE | re.DOTALL)

            for match in matches:
                line_number = text[:match.start()].count("\n") + 1
                snippet = match.group(0)
                snippet = snippet[:160].replace("\n", " ")

                findings.append({
                    "file": relative,
                    "line": line_number,
                    "name": pattern["name"],
                    "severity": pattern["severity"],
                    "description": pattern["description"],
                    "snippet": snippet
                })

        return findings

    def detect_sensitive_files(self, repo_dir):
        sensitive_names = {
            ".env",
            "id_rsa",
            "id_dsa",
            "id_ecdsa",
            "id_ed25519",
            "credentials.json",
            "credentials.yml",
            "credentials.yaml",
            "secrets.json",
            "secrets.yml",
            "secrets.yaml",
            "private.key",
            "server.key",
            "client.key"
        }

        allowed_example_patterns = [
            ".example",
            ".sample",
            "template",
            "dummy",
            "test",
            "mock"
        ]

        sensitive_files = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv"
                }
            ]

            for file in files:
                lower_file = file.lower()

                if lower_file in sensitive_names:
                    path = Path(root) / file
                    relative = str(path.relative_to(repo_dir))

                    if any(pattern in relative.lower() for pattern in allowed_example_patterns):
                        continue

                    sensitive_files.append(relative)

        return sensitive_files

    def detect_security_positive_evidence(self, repo_dir):
        positive_files = []
        positive_patterns = []

        candidate_files = [
            "SECURITY.md",
            "security.md",
            ".github/dependabot.yml",
            ".github/dependabot.yaml",
            ".github/workflows/codeql.yml",
            ".github/workflows/codeql.yaml",
            ".github/workflows/security.yml",
            ".github/workflows/security.yaml",
            ".pre-commit-config.yaml",
            "bandit.yml",
            ".bandit",
            ".gitignore"
        ]

        for candidate in candidate_files:
            if (repo_dir / candidate).exists():
                positive_files.append(candidate)

        gitignore_path = repo_dir / ".gitignore"
        if gitignore_path.exists():
            text = self.read_text_file(gitignore_path)
            for pattern in [".env", "*.key", "secrets", "credentials"]:
                if pattern in text:
                    positive_patterns.append(f".gitignore excludes {pattern}")

        return positive_files, positive_patterns

    def evaluate_secure_configuration(self, repo_dir, artifact_data):
        config = artifact_data.get("secure_configuration", {})

        minimum_score = config.get("minimum_score", self.minimum_score)
        max_high_severity_findings = config.get(
            "max_high_severity_findings",
            self.max_high_severity_findings
        )
        max_medium_severity_findings = config.get(
            "max_medium_severity_findings",
            self.max_medium_severity_findings
        )

        scan_files = self.collect_scan_files(repo_dir)

        all_findings = []

        for path in scan_files:
            all_findings.extend(self.detect_findings_in_file(repo_dir, path))

        sensitive_files = self.detect_sensitive_files(repo_dir)
        positive_files, positive_patterns = self.detect_security_positive_evidence(repo_dir)

        for sensitive_file in sensitive_files:
            all_findings.append({
                "file": sensitive_file,
                "line": None,
                "name": "sensitive_file_committed",
                "severity": "high",
                "description": "Sensitive file appears to be committed",
                "snippet": sensitive_file
            })

        high_findings = [f for f in all_findings if f["severity"] == "high"]
        medium_findings = [f for f in all_findings if f["severity"] == "medium"]
        low_findings = [f for f in all_findings if f["severity"] == "low"]

        score = 0
        evidence = []
        issues = []

        if len(high_findings) <= max_high_severity_findings:
            score += 2
            evidence.append(f"High-severity findings within threshold: {len(high_findings)} <= {max_high_severity_findings}")
        else:
            issues.append(f"High-severity findings exceed threshold: {len(high_findings)} > {max_high_severity_findings}")

        if len(medium_findings) <= max_medium_severity_findings:
            score += 1
            evidence.append(f"Medium-severity findings within threshold: {len(medium_findings)} <= {max_medium_severity_findings}")
        else:
            issues.append(f"Medium-severity findings exceed threshold: {len(medium_findings)} > {max_medium_severity_findings}")

        if not sensitive_files:
            score += 1
            evidence.append("No obvious sensitive files committed")
        else:
            issues.append(f"Sensitive files detected: {', '.join(sensitive_files[:10])}")

        if positive_files:
            score += 1
            evidence.append(f"Security-related files found: {', '.join(positive_files[:10])}")
        else:
            issues.append("No explicit security configuration files found")

        if positive_patterns:
            score += 1
            evidence.append(f"Protective ignore patterns found: {', '.join(positive_patterns[:10])}")
        else:
            issues.append("No protective .gitignore patterns for secrets detected")

        if len(scan_files) > 0:
            score += 1
            evidence.append(f"Scanned {len(scan_files)} relevant text/configuration files")
        else:
            issues.append("No relevant files available for secure-configuration scanning")

        secure_configuration = (
            score >= minimum_score
            and len(high_findings) <= max_high_severity_findings
            and len(medium_findings) <= max_medium_severity_findings
        )

        return {
            "secure_configuration": secure_configuration,
            "score": score,
            "minimum_score": minimum_score,
            "scanned_file_count": len(scan_files),
            "finding_count": len(all_findings),
            "high_findings_count": len(high_findings),
            "medium_findings_count": len(medium_findings),
            "low_findings_count": len(low_findings),
            "sensitive_files": sensitive_files,
            "positive_security_files": positive_files,
            "positive_security_patterns": positive_patterns,
            "findings": all_findings[:100],
            "evidence": evidence,
            "issues": issues
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Secure Configuration Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "secure_configuration": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_secure_configuration(repo_dir, artifact_data)
        artifact_result.update(result)

        print("\n📊 Secure configuration evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Scanned files: {result['scanned_file_count']}")
        print(f" - Total findings: {result['finding_count']}")
        print(f" - High findings: {result['high_findings_count']}")
        print(f" - Medium findings: {result['medium_findings_count']}")
        print(f" - Low findings: {result['low_findings_count']}")
        print(f" - Sensitive files: {', '.join(result['sensitive_files']) if result['sensitive_files'] else 'None'}")
        print(f" - Positive security files: {', '.join(result['positive_security_files']) if result['positive_security_files'] else 'None'}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No secure-configuration evidence found.")

        print("\n⚠️ Issues / weak evidence:")
        if result["issues"]:
            for item in result["issues"]:
                print(f" - {item}")
        else:
            print(" - No major secure-configuration issues detected.")

        if result["findings"]:
            print("\n🚩 Example findings:")
            for finding in result["findings"][:10]:
                location = finding["file"]
                if finding["line"] is not None:
                    location += f":{finding['line']}"
                print(f" - [{finding['severity']}] {location} — {finding['description']}")

        if result["secure_configuration"]:
            artifact_result["status"] = "passed"
            print("\n✅ Secure Configuration Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Secure Configuration Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Secure Configuration Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Secure Configuration Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["secure_configuration"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = SecureConfigurationChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    minimum_score=5,
    max_high_severity_findings=0,
    max_medium_severity_findings=5
)

secure_configuration_results = checker.run()

🌱 Starting Secure Configuration Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Secure Configuration Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Secure configuration evidence:
 - Score: 7 / required 5
 - Scanned files: 12
 - Total findings: 0
 - High findings: 0
 - Medium findings: 0
 - Low findings: 0
 - Sensitive files: None
 - Positive security files: .gitignore

🔎 Evidence found:
 - High-severity findings within threshold: 0 <= 0
 - Medium-severity findings within threshold: 0 <= 5
 - No obvious sensitive files committed
 - Security-related files found: .gitignore
 - Protective ignore patterns found: .gitignore excludes .env
 - Scanned 12 relevant text/configuration files

⚠️ Issues / weak evidence:
 - No major secure-configuration issues detected.

✅ Secure Configuration Result: PASSED

🔍 Secure Configuration Check for

In [4]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(secure_configuration_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "secure_configuration",
        "status",
        "score",
        "minimum_score",
        "scanned_file_count",
        "finding_count",
        "high_findings_count",
        "medium_findings_count",
        "low_findings_count",
        "positive_security_files"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in secure_configuration_results:
        print(result)

,artifact_id,title,secure_configuration,status,score,minimum_score,scanned_file_count,finding_count,high_findings_count,medium_findings_count,low_findings_count,positive_security_files
0,artifact_1,We provide our resources in a dedicated reposi...,True,passed,7.0,5.0,12.0,0.0,0.0,0.0,0.0,[.gitignore]
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,False,failed,4.0,5.0,1479.0,34.0,2.0,29.0,3.0,"[.github/dependabot.yml, .pre-commit-config.ya..."
3,artifact_4,Scikit-learn,False,failed,4.0,5.0,1387.0,255.0,1.0,241.0,13.0,"[SECURITY.md, .github/dependabot.yml, .github/..."
4,artifact_5,Pandas,False,failed,3.0,5.0,1825.0,491.0,4.0,481.0,6.0,"[.github/dependabot.yml, .github/workflows/cod..."
5,artifact_6,NumPy,False,failed,3.0,5.0,1172.0,290.0,8.0,242.0,40.0,"[.github/dependabot.yml, .github/workflows/cod..."
6,artifact_7,Matplotlib,False,failed,4.0,5.0,1525.0,189.0,1.0,176.0,12.0,"[SECURITY.md, .github/dependabot.yml, .pre-com..."
7,artifact_8,Scrapy,False,failed,3.0,5.0,536.0,1801.0,3.0,1795.0,3.0,"[SECURITY.md, .pre-commit-config.yaml, .gitign..."
8,artifact_9,Flask,False,failed,3.0,5.0,195.0,102.0,6.0,96.0,0.0,"[.pre-commit-config.yaml, .gitignore]"
9,artifact_10,TensorFlow,False,failed,3.0,5.0,4888.0,5255.0,4.0,5183.0,68.0,"[SECURITY.md, .github/dependabot.yml, .gitignore]"


In [5]:
output_file = "secure_configuration_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(secure_configuration_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to secure_configuration_results.json
